<a href="https://colab.research.google.com/github/aliasjad6536/AliAsjad/blob/main/heeal_matrix_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("📦 Installing packages... (this may take 2-3 minutes)")

!pip install -q langchain langchain-openai langgraph fastapi uvicorn gradio twilio pydantic requests nest-asyncio pyngrok gtts openai googlemaps pillow opencv-python-headless transformers torch torchvision deepface fer mediapipe

print("✅ All packages installed successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Installing packages... (this may take 2-3 minutes)
✅ All packages installed successfully!


In [ ]:
# CELL 2: Install and Setup Ollama (Local AI Model)
print("🔧 Installing Ollama...")
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start Ollama server
print("🚀 Starting Ollama server...")
ollama_process = subprocess.Popen(['ollama', 'serve'],
                                   stdout=subprocess.PIPE,
                                   stderr=subprocess.PIPE)
time.sleep(5)

# Download therapy model
print("📥 Downloading gemma2:2b model... (this may take a few minutes)")
!ollama pull gemma2:2b

print("✅ Ollama installed and gemma2:2b model ready!")

🔧 Installing Ollama...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🚀 Starting Ollama server...
📥 Downloading gemma2:2b model... (this may take a few minutes)

✅ Ollama installed and gemma2:2b model ready!


In [ ]:
# CELL 3: Configuration - ADD YOUR API KEYS HERE

OPENAI_API_KEY = ""

# Twilio Configuration (Required for calls & WhatsApp)
TWILIO_ACCOUNT_SID = ""
TWILIO_AUTH_TOKEN = ""
TWILIO_PHONE_NUMBER = "+18027121187"  # Your Twilio phone number
TWILIO_WHATSAPP_NUMBER = "whatsapp:+14155238886"  # Twilio WhatsApp Sandbox number
EMERGENCY_CONTACT = ""  # Emergency contact phone
EMERGENCY_WHATSAPP = "whatsapp:"  # Emergency WhatsApp

# Google Maps API Key (Required for doctor search)
GOOGLE_MAPS_API_KEY = ""
# File Storage Path (where data will be saved)
BASE_DRIVE_PATH = "/content/drive/MyDrive/SafeSpace_Data"

print("Validating configuration...\n")

if not OPENAI_API_KEY or OPENAI_API_KEY == "your-openai-api-key-here":
    print(" WARNING: Add your OpenAI API key!")
    print("   Get it from: https://platform.openai.com/api-keys\n")
else:
    print("OpenAI API key configured!\n")

if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "your-google-maps-api-key":
    print("WARNING: Add your Google Maps API key!")
    print("   Get it from: https://console.cloud.google.com/google/maps-apis")
    print("   Enable: Places API, Geocoding API, Maps JavaScript API\n")
else:
    print("Google Maps API key configured!\n")

if not TWILIO_ACCOUNT_SID or TWILIO_ACCOUNT_SID == "your-twilio-account-sid":
    print(" WARNING: Add your Twilio credentials for emergency features!")
    print("   Get from: https://www.twilio.com/console\n")
else:
    print(" Twilio credentials configured!\n")

print(f" Data will be saved to: {BASE_DRIVE_PATH}")
print("Configuration loaded!")

Validating configuration...

OpenAI API key configured!

   Get it from: https://console.cloud.google.com/google/maps-apis
   Enable: Places API, Geocoding API, Maps JavaScript API

 Twilio credentials configured!

 Data will be saved to: /content/drive/MyDrive/SafeSpace_Data
Configuration loaded!


In [ ]:
# CELL 4: File Storage System for Persistent Data
import os
import json
from datetime import datetime

# Create necessary directories
directories = [
    f"{BASE_DRIVE_PATH}/chat_logs",
    f"{BASE_DRIVE_PATH}/emotions",
    f"{BASE_DRIVE_PATH}/audio",
    f"{BASE_DRIVE_PATH}/crisis_alerts",
    f"{BASE_DRIVE_PATH}/session_data"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f" Created: {directory}")

def save_chat_log(user_message, ai_response, crisis_detected, tool_used):
    """Save complete chat interaction to file"""
    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    date_folder = datetime.now().strftime('%Y-%m-%d')

    # Create date-based subfolder
    folder_path = f"{BASE_DRIVE_PATH}/chat_logs/{date_folder}"
    os.makedirs(folder_path, exist_ok=True)

    filename = f"{folder_path}/chat_{timestamp}.json"

    chat_data = {
        "timestamp": timestamp,
        "user_message": user_message,
        "ai_response": ai_response,
        "crisis_detected": crisis_detected,
        "tool_used": tool_used,
        "date": date_folder
    }

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(chat_data, f, indent=2, ensure_ascii=False)

    print(f"Chat log saved: {filename}")
    return filename

def save_emotion_analysis(image_path, emotion_result, emotion_label, confidence):
    """Save emotion analysis with image"""
    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    date_folder = datetime.now().strftime('%Y-%m-%d')

    folder_path = f"{BASE_DRIVE_PATH}/emotions/{date_folder}"
    os.makedirs(folder_path, exist_ok=True)

    # Save image
    if image_path:
        import shutil
        image_filename = f"{folder_path}/image_{timestamp}.jpg"
        shutil.copy(image_path, image_filename)

    # Save analysis data
    data_filename = f"{folder_path}/analysis_{timestamp}.json"

    emotion_data = {
        "timestamp": timestamp,
        "emotion": emotion_label,
        "confidence": confidence,
        "full_result": emotion_result,
        "image_path": image_filename if image_path else None
    }

    with open(data_filename, 'w', encoding='utf-8') as f:
        json.dump(emotion_data, f, indent=2, ensure_ascii=False)

    print(f"Emotion analysis saved: {data_filename}")
    return data_filename

def save_crisis_alert(user_message, actions_taken):
    """Save crisis alert details"""
    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    filename = f"{BASE_DRIVE_PATH}/crisis_alerts/crisis_{timestamp}.json"

    crisis_data = {
        "timestamp": timestamp,
        "user_message": user_message,
        "actions_taken": actions_taken,
        "emergency_contact": EMERGENCY_CONTACT
    }

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(crisis_data, f, indent=2, ensure_ascii=False)

    print(f"Crisis alert saved: {filename}")
    return filename

def get_session_summary():
    """Get summary of all saved data"""
    summary = {
        "total_chats": len([f for f in os.listdir(f"{BASE_DRIVE_PATH}/chat_logs") if os.path.isdir(f"{BASE_DRIVE_PATH}/chat_logs/{f}")]),
        "total_emotions": len([f for f in os.listdir(f"{BASE_DRIVE_PATH}/emotions") if os.path.isdir(f"{BASE_DRIVE_PATH}/emotions/{f}")]),
        "total_crisis_alerts": len([f for f in os.listdir(f"{BASE_DRIVE_PATH}/crisis_alerts") if f.endswith('.json')]),
        "storage_path": BASE_DRIVE_PATH
    }
    return summary

print("File storage system initialized!")
print(f" Storage location: {BASE_DRIVE_PATH}")

 Created: /content/drive/MyDrive/SafeSpace_Data/chat_logs
 Created: /content/drive/MyDrive/SafeSpace_Data/emotions
 Created: /content/drive/MyDrive/SafeSpace_Data/audio
 Created: /content/drive/MyDrive/SafeSpace_Data/crisis_alerts
 Created: /content/drive/MyDrive/SafeSpace_Data/session_data
File storage system initialized!
 Storage location: /content/drive/MyDrive/SafeSpace_Data


In [ ]:
# CELL 5: Advanced Crisis Detection System
import re

CRISIS_KEYWORDS = [
    'suicide', 'suicidal', 'kill myself', 'end my life', 'want to die',
    'better off dead', 'no reason to live', 'cant go on', "can't go on",
    'self harm', 'self-harm', 'hurt myself', 'end it all', 'not worth living',
    'take my life', 'overdose', 'jump off', 'hang myself', 'cut myself',
    'want to disappear', 'wish i was dead', 'everyone would be better without me',
    'no point in living', 'ready to die', 'plan to die', 'thinking about dying'
]

def detect_crisis(text: str) -> bool:
    """
    Detect crisis keywords in user message
    Returns True if crisis detected
    """
    if not text:
        return False

    text_lower = text.lower()

    for keyword in CRISIS_KEYWORDS:
        if keyword in text_lower:
            print(f"CRISIS KEYWORD DETECTED: '{keyword}'")
            return True

    return False

def get_crisis_severity(text: str) -> str:
    """
    Determine crisis severity level
    Returns: 'high', 'medium', 'low', or 'none'
    """
    text_lower = text.lower()

    high_risk = ['kill myself', 'end my life', 'want to die', 'suicide', 'hang myself', 'overdose']
    medium_risk = ['self harm', 'hurt myself', 'no reason to live', 'better off dead']

    for keyword in high_risk:
        if keyword in text_lower:
            return 'high'

    for keyword in medium_risk:
        if keyword in text_lower:
            return 'medium'

    return 'none'

print("Crisis detection system initialized!")
print(f"Monitoring for {len(CRISIS_KEYWORDS)} crisis keywords")

Crisis detection system initialized!
Monitoring for 26 crisis keywords


In [ ]:
# CELL 6: Twilio for emeergency communications
from twilio.rest import Client

def call_emergency():
    """Place emergency phone call via Twilio"""
    try:
        client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        call = client.calls.create(
            to=EMERGENCY_CONTACT,
            from_=TWILIO_PHONE_NUMBER,
            url=""
        )
        print(f"Emergency call initiated: {call.sid}")
        return f"Emergency call placed successfully. Call SID: {call.sid}"
    except Exception as e:
        print(f"Failed to place call: {str(e)}")
        return f"Call failed: {str(e)}"

def send_whatsapp_emergency(message: str):
    """Send emergency WhatsApp alert"""
    try:
        client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        whatsapp_msg = client.messages.create(
            body=f" MENTAL HEALTH EMERGENCY ALERT \n\n{message}\n\nPlease check on this person immediately.\n\nTime: {timestamp}",
            from_=TWILIO_WHATSAPP_NUMBER,
            to=EMERGENCY_WHATSAPP
        )
        print(f" WhatsApp emergency sent: {whatsapp_msg.sid}")
        return f"WhatsApp alert sent. Message SID: {whatsapp_msg.sid}"
    except Exception as e:
        print(f"Failed to send WhatsApp: {str(e)}")
        return f"WhatsApp failed: {str(e)}"

def send_whatsapp_message(to_number: str, message: str):
    """Send regular WhatsApp message"""
    try:
        client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        msg = client.messages.create(
            body=message,
            from_=TWILIO_WHATSAPP_NUMBER,
            to=f"whatsapp:{to_number}"
        )
        print(f"📱 WhatsApp message sent: {msg.sid}")
        return f"WhatsApp message sent: {msg.sid}"
    except Exception as e:
        print(f" WhatsApp error: {str(e)}")
        return f"WhatsApp error: {str(e)}"

print("Twilio integration ready!")
print(f" Emergency contact: {EMERGENCY_CONTACT}")
print(f"WhatsApp alerts enabled")

Twilio integration ready!
 Emergency contact: +923123132321
WhatsApp alerts enabled


In [ ]:
# CELL 7: Google Maps Integration to findd Doctor Search

import googlemaps
import traceback

# CELL 7: Doctor Search (DISABLED - Static Resources)

def find_nearest_doctors(location: str, radius: int = 5000):
    """
    Returns helpful mental health resources without API
    No Google Maps API needed
    """
    return f"""**🔍 Mental Health Resources Near {location}**

Google Maps integration is disabled. Here are verified resources:

**🏥 Mental Health Facilities in Lahore:**

1. **Fountain House**
   📍 70-C/1, Township, Lahore
   📞 +92-42-35413744, +92-42-35414073
   ⭐ Leading mental health facility in Pakistan

2. **Willing Ways Psychiatric Rehabilitation Centre**
   📍 69-B, Jail Road, Lahore
   📞 +92-42-35912928, +92-42-35913673
   ⭐ Specializes in addiction and rehabilitation

3. **Institute of Behavioral Sciences**
   📍 New Garden Town, Lahore
   📞 +92-42-35913212
   ⭐ Clinical psychology and therapy services

4. **Jinnah Hospital Psychiatry Department**
   📍 Allama Shabbir Ahmed Usmani Road, Lahore
   📞 +92-42-99231441
   ⭐ Government facility with affordable care

5. **Karachi Psychiatric Hospital**
   📍 Karachi
   📞 +92-21-99230231
   ⭐ Major facility in Karachi

**📞 Mental Health Helplines:**
- Pakistan Mental Health Helpline: **0800-00-002** (Toll-free)
- Rozan Helpline: **0800-22-444** (For women)
- Umang Helpline: **0317-4288665** (Youth support)
- Emergency Services: **1122**

**🌐 Online Directories:**
- Marham.pk: https://www.marham.pk/doctors/lahore/psychiatrist
- Oladoc: https://oladoc.com/pakistan/lahore/psychiatrist
- Psychology Today: https://www.psychologytoday.com/pk/therapists

**💡 Search Manually:**
- Google: "psychiatrist near {location}"
- Google Maps: "therapist near me"
- Search: "{location} mental health services"

**🌍 If you're in another city:**
Replace "Lahore" with your city name when searching online.

Would you like help with anything else?"""

print("✅ Doctor search configured (static resources - no API needed)")

✅ Doctor search configured (static resources - no API needed)


**cell 8 change**

In [ ]:
# ================================
# CELL 8: Emotion Detection (FIXED)
# ================================

print("🔄 Installing CV libraries...")

!pip uninstall -y tensorflow keras protobuf fer deepface -q

!pip install -q tensorflow deepface opencv-python-headless pillow transformers torch torchvision timm

print("✅ Installation complete!")

# ================================
# IMPORTS
# ================================
import torch
import numpy as np
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# DeepFace
try:
    from deepface import DeepFace
    DEEPFACE_AVAILABLE = True
    print("✅ DeepFace loaded!")
except Exception as e:
    DEEPFACE_AVAILABLE = False
    print(f"⚠️ DeepFace not available: {e}")

# HuggingFace
try:
    from transformers import pipeline
    emotion_classifier = pipeline(
        "image-classification",
        model="trpakov/vit-face-expression",
        device=0 if torch.cuda.is_available() else -1
    )
    HF_AVAILABLE = True
    print("✅ HuggingFace model loaded!")
except Exception as e:
    HF_AVAILABLE = False
    print(f"⚠️ HuggingFace not available: {e}")

# ================================
# MODEL 1: DeepFace
# ================================
def analyze_emotion_deepface(image_path):
    if not DEEPFACE_AVAILABLE:
        return None, 0, {}
    try:
        result = DeepFace.analyze(
            img_path=image_path,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend='opencv',
            silent=True
        )
        if isinstance(result, list):
            result = result[0]
        emotions = result['emotion']
        dominant = result['dominant_emotion']
        confidence = emotions[dominant]
        return dominant, confidence, emotions
    except Exception as e:
        print(f"DeepFace error: {e}")
        return None, 0, {}

# ================================
# MODEL 2: HuggingFace
# ================================
def analyze_emotion_hf(image_path):
    if not HF_AVAILABLE:
        return None, 0, {}
    try:
        image = Image.open(image_path).convert("RGB")
        results = emotion_classifier(image)
        top = results[0]
        emotion = top['label']
        confidence = top['score'] * 100
        emotions = {r['label']: r['score'] * 100 for r in results}
        return emotion, confidence, emotions
    except Exception as e:
        print(f"HF error: {e}")
        return None, 0, {}

# ================================
# NORMALIZATION
# ================================
def normalize(emotion):
    mapping = {
        'angry': 'angry', 'anger': 'angry',
        'disgust': 'disgust',
        'fear': 'fear', 'scared': 'fear',
        'happy': 'happy', 'happiness': 'happy',
        'sad': 'sad', 'sadness': 'sad',
        'surprise': 'surprise', 'surprised': 'surprise',
        'neutral': 'neutral'
    }
    return mapping.get(emotion.lower(), emotion.lower())

# ================================
# ENSEMBLE SYSTEM
# ================================
def ensemble_emotion_detection(image_path):
    print("\n" + "="*50)
    print("🧠 MULTI-MODEL ANALYSIS")
    print("="*50)

    predictions = []

    e1, c1, _ = analyze_emotion_deepface(image_path)
    if e1:
        e1 = normalize(e1)
        predictions.append((e1, c1, "DeepFace"))
        print(f"DeepFace → {e1} ({c1:.1f}%)")

    e2, c2, _ = analyze_emotion_hf(image_path)
    if e2:
        e2 = normalize(e2)
        predictions.append((e2, c2, "HuggingFace"))
        print(f"HuggingFace → {e2} ({c2:.1f}%)")

    if not predictions:
        return None, 0, []

    scores = {}
    for e, c, _ in predictions:
        scores.setdefault(e, []).append(c)

    final_scores = {}
    for e, vals in scores.items():
        avg = sum(vals) / len(vals)
        weight = len(vals) / len(predictions)
        final_scores[e] = avg * (1 + weight)

    final_emotion = max(final_scores, key=final_scores.get)
    final_conf = final_scores[final_emotion]

    return final_emotion, final_conf, predictions

# ================================
# MAIN FUNCTION
# ================================
def analyze_facial_emotion(image_path):
    try:
        img = Image.open(image_path)
        print(f"📷 Image Loaded: {img.size}")

        emotion, confidence, preds = ensemble_emotion_detection(image_path)

        if not emotion:
            return "❌ No face detected.", "error", 0

        responses = {
            'sad': "You seem sad. I'm here for you.",
            'angry': "I sense anger. Want to talk?",
            'happy': "You look happy! 😊",
            'fear': "You seem anxious. You're safe here.",
            'surprise': "You look surprised!",
            'disgust': "Something seems uncomfortable.",
            'neutral': "I'm here to listen."
        }

        result = f"✅ EMOTION ANALYSIS\n\nEmotion: {emotion.upper()}\nConfidence: {confidence:.1f}%\n\nModel Results:\n"
        for e, c, m in preds:
            result += f"- {m}: {e} ({c:.1f}%)\n"
        result += f"\nResponse: {responses.get(emotion, 'I am here for you.')}"

        return result, emotion, confidence

    except Exception as e:
        return f"Error: {e}", "error", 0

# ================================
# IMAGE QUALITY CHECK
# ================================
def check_image_quality(image_path):
    try:
        img = cv2.imread(image_path)
        if img is None:
            return False, "Invalid image"
        h, w = img.shape[:2]
        if h < 100 or w < 100:
            return False, "Image too small"
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness = np.mean(gray)
        if brightness < 30:
            return False, "Too dark"
        if brightness > 225:
            return False, "Too bright"
        return True, "Good image"
    except Exception as e:
        return True, f"Quality check skipped: {e}"

print("\n" + "="*50)
print("🚀 SYSTEM READY!")
print(f"DeepFace: {'✅' if DEEPFACE_AVAILABLE else '❌'}")
print(f"HuggingFace: {'✅' if HF_AVAILABLE else '❌'}")
print("="*50)

🔄 Installing CV libraries...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.21.0 which is incompatible.
google-cloud-resource-manager 1.16.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-api-core 2.30.0 requires protobuf<7.0.0,>=4.25.8, but you have protobuf 7.34.1 which is incompatible.
google-cloud-dataplex 2.16.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-trace 1.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,

config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ HuggingFace model loaded!

🚀 SYSTEM READY!
DeepFace: ✅
HuggingFace: ✅


In [ ]:
!pip install mediapipe

In [ ]:
# cell 8.5 pose detetcion
import mediapipe as mp
import cv2
print("MediaPipe version:", mp.__version__)
print("OpenCV version:", cv2.__version__)


MediaPipe version: 0.10.33
OpenCV version: 4.13.0


In [ ]:
# CELL 8.5: Pose Detection with MediaPipe (FIXED)

print("📦 Installing MediaPipe and OpenCV...")
!pip install -q mediapipe opencv-python-headless pillow

import mediapipe as mp
import cv2
import numpy as np
from PIL import Image
import math
import warnings
import os
from datetime import datetime
warnings.filterwarnings('ignore')

print("✅ Imports successful!")

# Initialize MediaPipe Pose
print("🔧 Initializing MediaPipe Pose...")

try:
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    pose_detector = mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,
        enable_segmentation=False,
        min_detection_confidence=0.5
    )

    print("✅ MediaPipe Pose initialized!")
    POSE_AVAILABLE = True

except AttributeError as e:
    print(f"⚠️ MediaPipe Pose not available: {e}")
    print("⚠️ Pose detection will be disabled")
    POSE_AVAILABLE = False
    mp_pose = None
    mp_drawing = None
    mp_drawing_styles = None
    pose_detector = None

# Helper Functions
def calculate_angle(a, b, c):
    """Calculate angle between three points"""
    try:
        a = np.array([a.x, a.y])
        b = np.array([b.x, b.y])
        c = np.array([c.x, c.y])

        radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
        angle = np.abs(radians * 180.0 / np.pi)

        if angle > 180.0:
            angle = 360 - angle

        return angle
    except:
        return 0

def calculate_distance(point1, point2):
    """Calculate Euclidean distance between two points"""
    try:
        return math.sqrt((point1.x - point2.x)**2 + (point1.y - point2.y)**2)
    except:
        return 0

def detect_head_tilt(landmarks):
    """Detect head tilt angle"""
    try:
        left_ear = landmarks[mp_pose.PoseLandmark.LEFT_EAR.value]
        right_ear = landmarks[mp_pose.PoseLandmark.RIGHT_EAR.value]
        nose = landmarks[mp_pose.PoseLandmark.NOSE.value]

        ear_midpoint_y = (left_ear.y + right_ear.y) / 2
        head_tilt = abs(nose.y - ear_midpoint_y)
        lr_tilt = abs(left_ear.y - right_ear.y)

        if head_tilt > 0.15:
            return "head_down", head_tilt
        elif lr_tilt > 0.08:
            return "head_tilted", lr_tilt
        else:
            return "neutral", 0
    except:
        return "neutral", 0

def detect_shoulder_posture(landmarks):
    """Detect shoulder posture"""
    try:
        left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
        right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
        left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
        right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]

        shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
        hip_y = (left_hip.y + right_hip.y) / 2
        torso_length = abs(hip_y - shoulder_y)

        if torso_length < 0.35:
            return "slouched", torso_length
        elif torso_length > 0.45:
            return "upright", torso_length
        else:
            return "normal", torso_length
    except:
        return "normal", 0.4

def detect_arm_position(landmarks):
    """Detect arm positions"""
    try:
        left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
        right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]
        left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
        right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
        nose = landmarks[mp_pose.PoseLandmark.NOSE.value]

        left_wrist_to_right_shoulder = calculate_distance(left_wrist, right_shoulder)
        right_wrist_to_left_shoulder = calculate_distance(right_wrist, left_shoulder)

        if left_wrist_to_right_shoulder < 0.3 or right_wrist_to_left_shoulder < 0.3:
            return "crossed", "defensive"

        left_wrist_to_face = calculate_distance(left_wrist, nose)
        right_wrist_to_face = calculate_distance(right_wrist, nose)

        if left_wrist_to_face < 0.15 or right_wrist_to_face < 0.15:
            return "covering_face", "anxious"

        return "neutral", "relaxed"
    except:
        return "neutral", "relaxed"

def analyze_overall_body_language(head_state, shoulder_state, arm_state):
    """Combine all pose indicators"""
    body_language_insights = {
        "posture_score": 0,
        "stress_indicators": [],
        "confidence_level": "neutral",
        "emotional_state": "neutral",
        "recommendations": []
    }

    if head_state[0] == "head_down":
        body_language_insights["posture_score"] -= 3
        body_language_insights["stress_indicators"].append("Head tilted down (possible sadness/fatigue)")
        body_language_insights["emotional_state"] = "low_mood"
    elif head_state[0] == "head_tilted":
        body_language_insights["stress_indicators"].append("Head tilted (possible confusion/concern)")

    if shoulder_state[0] == "slouched":
        body_language_insights["posture_score"] -= 4
        body_language_insights["stress_indicators"].append("Slouched shoulders (low confidence/energy)")
        body_language_insights["confidence_level"] = "low"
        body_language_insights["recommendations"].append("Try gentle shoulder rolls to release tension")
    elif shoulder_state[0] == "upright":
        body_language_insights["posture_score"] += 3
        body_language_insights["confidence_level"] = "high"

    if arm_state[0] == "crossed":
        body_language_insights["posture_score"] -= 2
        body_language_insights["stress_indicators"].append("Arms crossed (defensive/closed-off)")
        body_language_insights["recommendations"].append("Consider relaxing your arms")
    elif arm_state[0] == "covering_face":
        body_language_insights["posture_score"] -= 5
        body_language_insights["stress_indicators"].append("Hands near face (high stress/anxiety)")
        body_language_insights["emotional_state"] = "anxious"
        body_language_insights["recommendations"].append("Take deep breaths and lower your hands")

    score = body_language_insights["posture_score"]
    if score >= 5:
        body_language_insights["overall_assessment"] = "positive"
        body_language_insights["message"] = "Your body language shows confidence and openness! 😊"
    elif score >= 0:
        body_language_insights["overall_assessment"] = "neutral"
        body_language_insights["message"] = "Your posture is fairly neutral."
    elif score >= -5:
        body_language_insights["overall_assessment"] = "some_concern"
        body_language_insights["message"] = "I notice some tension in your posture."
    else:
        body_language_insights["overall_assessment"] = "significant_concern"
        body_language_insights["message"] = "Your body language suggests you might be struggling."

    return body_language_insights

# Main Pose Detection Function
def analyze_body_language(image_path):
    """Complete pose detection and analysis"""

    # Check if pose detection is available
    if not POSE_AVAILABLE:
        return (
            "⚠️ **Pose Detection Not Available**\n\n"
            "Body language analysis is currently disabled due to MediaPipe compatibility issues.\n\n"
            "**What you can still use:**\n"
            "- 💬 AI Chat (fully functional)\n"
            "- 📸 Emotion Detection (fully functional)\n"
            "- 🎤 Voice Input (fully functional)\n"
            "- 🗺️ Doctor Search (fully functional)\n\n"
            "The emotion detection feature can still provide insights about your emotional state!",
            None,
            {}
        )

    try:
        print("\n" + "="*60)
        print("🧍 BODY LANGUAGE ANALYSIS STARTING")
        print("="*60)

        # Load image
        image = cv2.imread(image_path)
        if image is None:
            print(f"❌ Failed to load image: {image_path}")
            return "❌ Could not load image. Please try a different file.", None, {}

        print(f"✅ Image loaded: {image.shape}")

        # Resize if too large
        max_dimension = 1024
        height, width = image.shape[:2]
        if max(height, width) > max_dimension:
            scale = max_dimension / max(height, width)
            new_width = int(width * scale)
            new_height = int(height * scale)
            image = cv2.resize(image, (new_width, new_height))
            print(f"📐 Resized to: {image.shape}")

        # Convert BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Process with MediaPipe
        print("🔍 Processing with MediaPipe Pose...")
        results = pose_detector.process(image_rgb)

        if not results.pose_landmarks:
            print("❌ No person detected")
            return (
                "❌ **No person detected in image**\n\n"
                "**Tips for better results:**\n"
                "- Ensure full upper body is visible\n"
                "- Stand/sit 3-6 feet from camera\n"
                "- Use good lighting\n"
                "- Face the camera directly",
                None,
                {}
            )

        print("✅ Pose landmarks detected!")

        # Get landmarks
        landmarks = results.pose_landmarks.landmark

        # Analyze body parts
        print("📊 Analyzing body language...")
        head_state = detect_head_tilt(landmarks)
        shoulder_state = detect_shoulder_posture(landmarks)
        arm_state = detect_arm_position(landmarks)

        print(f"   👤 Head: {head_state[0]}")
        print(f"   💪 Shoulders: {shoulder_state[0]}")
        print(f"   🙌 Arms: {arm_state[0]}")

        # Get overall analysis
        body_data = analyze_overall_body_language(head_state, shoulder_state, arm_state)
        print(f"   📊 Overall Score: {body_data['posture_score']}/10")

        # Create annotated image
        print("🎨 Creating annotated image...")
        annotated_image = image.copy()
        mp_drawing.draw_landmarks(
            annotated_image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )

        # Save annotated image
        timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
        date_folder = datetime.now().strftime('%Y-%m-%d')
        pose_folder = f"{BASE_DRIVE_PATH}/emotions/{date_folder}"

        os.makedirs(pose_folder, exist_ok=True)

        annotated_path = f"{pose_folder}/pose_{timestamp}_annotated.jpg"
        cv2.imwrite(annotated_path, annotated_image)
        print(f"💾 Annotated image saved: {annotated_path}")

        # Format result
        result = f"""**🧍 BODY LANGUAGE ANALYSIS COMPLETE**

**Overall Assessment:** {body_data['overall_assessment'].replace('_', ' ').title()}
**Posture Score:** {body_data['posture_score']}/10
**Confidence Level:** {body_data['confidence_level'].title()}

**📋 Detected Indicators:**
"""

        if body_data['stress_indicators']:
            for indicator in body_data['stress_indicators']:
                result += f"• {indicator}\n"
        else:
            result += "• ✅ No significant stress indicators detected\n"

        result += f"\n**💭 Interpretation:**\n{body_data['message']}\n"

        if body_data['recommendations']:
            result += f"\n**💡 Suggestions:**\n"
            for rec in body_data['recommendations']:
                result += f"• {rec}\n"

        result += f"\n**🔍 Detailed Analysis:**\n"
        result += f"• **Head Position:** {head_state[0].replace('_', ' ').title()}\n"
        result += f"• **Shoulder Posture:** {shoulder_state[0].title()}\n"
        result += f"• **Arm Position:** {arm_state[0].replace('_', ' ').title()}\n"

        print("="*60)
        print("✅ ANALYSIS COMPLETE")
        print("="*60)

        return result, annotated_path, body_data

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return f"❌ Error analyzing pose: {str(e)}", None, {}

# Status message
print("\n" + "="*60)
if POSE_AVAILABLE:
    print("✅ POSE DETECTION SYSTEM READY!")
    print("="*60)
    print(f"🎯 MediaPipe Pose v{mp.__version__} loaded")
    print("🔍 33 body keypoints tracking enabled")
else:
    print("⚠️ POSE DETECTION DISABLED")
    print("="*60)
    print("⚠️ MediaPipe Pose not available on this system")
    print("✅ Other features (Chat, Emotion, Voice) still work!")
print("="*60)

📦 Installing MediaPipe and OpenCV...
✅ Imports successful!
🔧 Initializing MediaPipe Pose...
⚠️ MediaPipe Pose not available: module 'mediapipe' has no attribute 'solutions'
⚠️ Pose detection will be disabled

⚠️ POSE DETECTION DISABLED
⚠️ MediaPipe Pose not available on this system
✅ Other features (Chat, Emotion, Voice) still work!


In [ ]:
# CELL 8.5: Pose Detection (Disabled - Compatibility Mode)

print("⚠️ Pose detection disabled for compatibility")
print("✅ Other features (Chat, Emotion, Voice) fully functional")

POSE_AVAILABLE = False

def analyze_body_language(image_path):
    """Pose detection placeholder"""
    return (
        "⚠️ **Body Language Analysis Not Available**\n\n"
        "Pose detection is disabled on this system for compatibility reasons.\n\n"
        "**What you can still use:**\n"
        "- 💬 **AI Chat** - Full therapy conversations\n"
        "- 📸 **Emotion Detection** - Facial emotion analysis (85-95% accuracy)\n"
        "- 🎤 **Voice Input** - Speak naturally with voice responses\n"
        "- 🗺️ **Doctor Search** - Find therapists near you\n"
        "- 📱 **WhatsApp** - Contact friends/emergency contacts\n"
        "- 🚨 **Crisis Detection** - Automatic emergency alerts\n\n"
        "Use the emotion detection feature for insights about your emotional state!",
        None,
        {}
    )

print("✅ Pose detection placeholder configured")

⚠️ Pose detection disabled for compatibility
✅ Other features (Chat, Emotion, Voice) fully functional
✅ Pose detection placeholder configured


In [ ]:
# CELL 9 (ALTERNATIVE): Therapy AI WITHOUT Ollama (GPT-4 Only)
import requests as req
from openai import OpenAI
def query_medgemma(prompt: str) -> str:
    """
    Fallback: Uses GPT-4o-mini directly for therapy responses
    (Use this if Ollama keeps crashing)
    """
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)

        system_prompt = """You are Dr. Emily Hartman, a warm and experienced clinical psychologist with 15 years of practice.

Your approach:
1. **Emotional Attunement**: Recognize and validate emotions
2. **Gentle Normalization**: Help patients feel less alone
3. **Practical Guidance**: Offer actionable strategies
4. **Strengths-Focused**: Highlight resilience

Communication style:
- Use natural, conversational language
- Show genuine empathy and care
- Match the patient's emotional tone

Remember: You're here to support, not diagnose. Always prioritize emotional safety."""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            max_tokens=400,
            temperature=0.7
        )

        ai_response = response.choices[0].message.content.strip()
        print(f"✅ Therapy AI response generated ({len(ai_response)} chars)")
        return ai_response

    except Exception as e:
        print(f"❌ Therapy AI error: {str(e)}")
        return "I'm having technical difficulties, but your feelings matter. Please continue sharing, and I'll do my best to support you."

print("✅ Therapy AI configured (GPT-4o-mini fallback mode)")

✅ Therapy AI configured (GPT-4o-mini fallback mode)


In [ ]:
# CELL 10 - WORKING VERSION (COPY THIS ENTIRE CELL)

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

@tool
def ask_mental_health_specialist(query: str) -> str:
    """
    Generate therapeutic response using therapy AI model.
    Use for general mental health queries and emotional support.
    """
    return query_medgemma(query)

@tool
def emergency_call_tool() -> str:
    """
    🚨 EMERGENCY RESPONSE: Place immediate phone call AND send WhatsApp alert.
    USE IMMEDIATELY for suicidal thoughts or self-harm.
    """
    call_result = call_emergency()
    whatsapp_result = send_whatsapp_emergency("User expressed suicidal/self-harm thoughts.")
    crisis_file = save_crisis_alert("Crisis detected", f"{call_result} | {whatsapp_result}")

    return f"🚨 EMERGENCY RESPONSE ACTIVATED:\n\n📞 {call_result}\n📱 {whatsapp_result}\n💾 Alert saved to: {crisis_file}\n\n✅ Emergency contacts notified."

@tool
def find_nearby_mental_health_doctors(location: str) -> str:
    """
    Find nearest mental health professionals using Google Maps.
    Use when user asks for therapists, psychiatrists, counselors near a location.

    Args:
        location: City or area name (e.g., "Lahore", "Karachi", "Islamabad")
    """
    try:
        return find_nearest_doctors(location)
    except Exception as e:
        print(f"❌ Tool error: {str(e)}")
        return f"""⚠️ Unable to search for doctors right now.

Error: {str(e)}

**Please try:**
- Google Maps: Search "therapist near me"
- https://www.psychologytoday.com/pk/therapists
- Call: Pakistan Mental Health Helpline 0800-00-002"""

@tool
def send_whatsapp_to_friend(phone_number: str, message: str) -> str:
    """
    Send WhatsApp message to user's friend/family for support.
    """
    return send_whatsapp_message(phone_number, message)

# Create AI Agent
tools = [
    ask_mental_health_specialist,
    emergency_call_tool,
    find_nearby_mental_health_doctors,
    send_whatsapp_to_friend
]

print("🔧 Initializing GPT-4 powered AI agent...")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, api_key=OPENAI_API_KEY)
graph = create_react_agent(llm, tools=tools)

SYSTEM_PROMPT = """You are an advanced AI mental health support system with warmth, empathy, and vigilance.

**Your Mission:** Provide compassionate mental health support while ensuring user safety.

**Available Tools:**
1. `ask_mental_health_specialist` - Use for emotional support, therapy conversations
2. `find_nearby_mental_health_doctors` - Use when user needs local professional help
3. `send_whatsapp_to_friend` - Use when user wants to reach trusted contacts
4. `emergency_call_tool` - 🚨 USE IMMEDIATELY for suicidal thoughts or self-harm

**CRITICAL EMERGENCY PROTOCOL:**
⚠️ If you detect ANY keywords like:
- "I want to die", "kill myself", "end my life", "suicide"
- "hurt myself", "self-harm", "cut myself"

**ACTION:** IMMEDIATELY call `emergency_call_tool` FIRST, THEN provide crisis support.

**Guidelines:**
- Be warm, empathetic, and non-judgmental
- Ask open-ended questions
- Validate emotions before offering solutions
- ALWAYS prioritize life preservation
- Never minimize serious mental health concerns

Remember: You're a supportive companion, not a replacement for professional care."""

def parse_response(stream):
    """Extract tool name and response from agent stream"""
    tool_called_name = "None"
    final_response = None

    for s in stream:
        tool_data = s.get('tools')
        if tool_data:
            tool_messages = tool_data.get('messages')
            if tool_messages and isinstance(tool_messages, list):
                for msg in tool_messages:
                    tool_called_name = getattr(msg, 'name', 'None')

        agent_data = s.get('agent')
        if agent_data:
            messages = agent_data.get('messages')
            if messages and isinstance(messages, list):
                for msg in messages:
                    if msg.content:
                        final_response = msg.content

    return tool_called_name, final_response

print("✅ AI Agent initialized with 4 tools!")
print("🤖 Model: GPT-4o-mini")
print("🛠️ Tools: Therapy AI, Emergency Call, Doctor Search, WhatsApp")

🔧 Initializing GPT-4 powered AI agent...
✅ AI Agent initialized with 4 tools!
🤖 Model: GPT-4o-mini
🛠️ Tools: Therapy AI, Emergency Call, Doctor Search, WhatsApp


In [ ]:
# CELL 12: Voice Processing - Speech-to-Text & Text-to-Speech
from gtts import gTTS
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def transcribe_audio(audio_path):
    """
    Convert speech to text using OpenAI Whisper API
    Supports multiple languages and accents
    """
    if audio_path is None:
        return ""

    try:
        print(f"Transcribing audio: {audio_path}")

        with open(audio_path, "rb") as audio_file:
            transcript = openai_client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                language="en"
            )

        transcribed_text = transcript.text
        print(f"Transcription complete: {transcribed_text[:100]}...")
        return transcribed_text

    except Exception as e:
        print(f"Transcription error: {str(e)}")
        return ""

def text_to_speech(text):
    """
    Convert text to speech using Google Text-to-Speech
    Saves audio to persistent storage
    """
    try:
        # Clean text for TTS
        clean_text = text.split('\n\n_[Tool used:')[0]
        clean_text = clean_text.split('**CRISIS SUPPORT')[0]

        if not clean_text.strip():
            return None

        print(f" Generating speech for text ({len(clean_text)} chars)")

        # Generate TTS
        tts = gTTS(text=clean_text, lang='en', slow=False)

        # Save to persistent location with timestamp
        timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
        audio_path = f"{BASE_DRIVE_PATH}/audio/response_{timestamp}.mp3"
        tts.save(audio_path)

        print(f"Audio saved to: {audio_path}")
        return audio_path

    except Exception as e:
        print(f"TTS error: {str(e)}")
        return None

print("Voice processing ready!")
print("Speech-to-Text: OpenAI Whisper")
print("Text-to-Speech: Google TTS")


Voice processing ready!
Speech-to-Text: OpenAI Whisper
Text-to-Speech: Google TTS


In [ ]:
# ============================================================================
# CELL 13: Gradio Interface (DIRECT - No Backend)
# ============================================================================

import gradio as gr
from gtts import gTTS
from datetime import datetime

print("🚀 Starting Gradio interface (direct mode)...")

# ========================================
# DIRECT INTERFACE FUNCTIONS (No Backend)
# ========================================

def chat_with_ai(message, history):
    """Chat function - calls AI directly"""
    if not message.strip():
        return history, "", None

    try:
        print(f"\n💬 User: {message[:100]}...")
        history = history + [[message, "🤔 Thinking..."]]

        # Detect crisis
        is_crisis = detect_crisis(message)
        crisis_severity = get_crisis_severity(message)

        # Get AI response directly (no backend!)
        ai_response = query_medgemma(message)
        tool_used = "ask_mental_health_specialist"

        # Handle crisis
        if is_crisis:
            print(f"🚨 CRISIS DETECTED ({crisis_severity})")
            try:
                call_result = call_emergency()
                whatsapp_result = send_whatsapp_emergency(f"Crisis: {message[:200]}")
                crisis_file = save_crisis_alert(message, f"{call_result} | {whatsapp_result}")

                ai_response = (
                    f"🚨 **CRISIS SUPPORT ACTIVATED** 🚨\n\n"
                    f"{ai_response}\n\n"
                    f"📞 {call_result}\n"
                    f"📱 {whatsapp_result}\n\n"
                    f"**IMMEDIATE HELP:**\n"
                    f"- 🚨 Emergency: 911\n"
                    f"- ☎️ Suicide Prevention: 988\n"
                    f"- 📱 Text: HOME to 741741\n\n"
                    f"💙 You are not alone."
                )
                tool_used = "emergency_call_tool"
            except Exception as e:
                print(f"⚠️ Emergency error: {e}")

        # Save chat
        try:
            save_chat_log(message, ai_response, is_crisis, tool_used)
        except:
            pass

        # Generate voice
        audio_path = text_to_speech(ai_response)

        # Update history
        if tool_used != "ask_mental_health_specialist":
            full_response = f"{ai_response}\n\n---\n*🛠️ Tool: {tool_used}*"
        else:
            full_response = ai_response

        history[-1] = [message, full_response]
        print(f"✅ Response generated")
        return history, "", audio_path

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        history[-1] = [message, error_msg]
        return history, "", None

def process_audio_input(audio, history):
    """Voice input - transcribe and chat"""
    if audio is None:
        return history, "", None

    try:
        print("🎤 Processing voice...")
        history = history + [["🎤 [Voice]", "📝 Transcribing..."]]

        # Transcribe
        message = transcribe_audio(audio)
        if not message:
            history[-1] = ["🎤 [Voice]", "❌ Transcription failed"]
            return history, "", None

        print(f"✅ Transcribed: {message[:100]}...")
        history[-1] = [f"🎤 \"{message}\"", "🤔 Thinking..."]

        # Get response
        ai_response = query_medgemma(message)
        audio_path = text_to_speech(ai_response)

        # Save
        try:
            save_chat_log(message, ai_response, False, "voice_input")
        except:
            pass

        history[-1] = [f"🎤 \"{message}\"", ai_response]
        return history, "", audio_path

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        history = history + [["🎤 [Voice]", error_msg]]
        return history, "", None

def analyze_emotion_image(image, history):
    """Emotion analysis - direct function call"""
    if image is None:
        return history, ""

    try:
        print("📸 Analyzing emotion...")
        history = history + [["📸 [Photo]", "🔍 Analyzing..."]]

        # Check quality
        is_good, quality_msg = check_image_quality(image)
        if not is_good:
            history[-1] = ["📸 [Photo]", f"⚠️ {quality_msg}"]
            return history, ""

        # Analyze
        result, emotion, confidence = analyze_facial_emotion(image)

        # Save
        try:
            save_emotion_analysis(image, result, emotion, confidence)
        except:
            pass

        # Emoji
        emojis = {'happy': '😊', 'sad': '😢', 'angry': '😠',
                  'fear': '😰', 'surprise': '😲', 'disgust': '😖', 'neutral': '😐'}
        emoji = emojis.get(emotion, '🤔')

        history[-1] = ["📸 [Photo]", f"{emoji} {result}"]
        print(f"✅ Emotion: {emotion} ({confidence:.1f}%)")
        return history, ""

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        history = history + [["📸 [Photo]", error_msg]]
        return history, ""

def analyze_pose_image(image, history):
    """Pose analysis - direct function call"""
    if image is None:
        return history, "", None

    try:
        print("🧍 Analyzing pose...")
        history = history + [["🧍 [Pose]", "🔍 Analyzing..."]]

        # Analyze
        result, annotated, body_data = analyze_body_language(image)

        history[-1] = ["🧍 [Pose]", result]
        print("✅ Pose complete")
        return history, "", annotated

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        history = history + [["🧍 [Pose]", error_msg]]
        return history, "", None

def analyze_complete_image(image, history):
    """Combined analysis - direct function calls"""
    if image is None:
        return history, "", None

    try:
        print("🎯 Complete analysis...")
        history = history + [["🎯 [Complete]", "🔍 Analyzing..."]]

        # Both analyses
        emotion_result, emotion, confidence = analyze_facial_emotion(image)
        pose_result, annotated, body_data = analyze_body_language(image)

        combined = f"**🎯 COMPREHENSIVE ANALYSIS**\n\n{emotion_result}\n\n---\n\n{pose_result}"

        history[-1] = ["🎯 [Complete]", combined]
        return history, "", annotated

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        history = history + [["🎯 [Complete]", error_msg]]
        return history, "", None

def get_session_stats():
    """Get statistics"""
    try:
        stats = get_session_summary()
        return f"""## 📊 SafeSpace Statistics

### 💾 Data Summary
- **💬 Chats:** {stats.get('total_chats', 0)}
- **📸 Emotions:** {stats.get('total_emotions', 0)}
- **🚨 Crisis Alerts:** {stats.get('total_crisis_alerts', 0)}

### 📁 Storage
- **Location:** `{stats.get('storage_path', 'N/A')}`
- **Status:** ✅ Google Drive

### 📈 System
- **Mode:** ✅ Direct (No Backend)
- **AI:** ✅ GPT-4o-mini

---
*Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*
"""
    except Exception as e:
        return f"❌ Error: {str(e)}"

def clear_chat():
    """Clear chat"""
    return [], None

def show_welcome():
    """Welcome message"""
    return [[None, """👋 **Welcome to SafeSpace AI Therapist!**

I'm here to provide compassionate mental health support 24/7.

**Features:**
- 💬 Chat about anything
- 🎤 Voice conversations
- 📸 Emotion analysis
- 🧍 Body language analysis
- 🚨 Crisis detection

**Start by typing a message below!**

**Emergency:** If in crisis, call 911 or 988 immediately.
"""]]

# ========================================
# GRADIO INTERFACE
# ========================================

custom_css = """
.gradio-container {
    font-family: 'Inter', sans-serif;
    max-width: 1400px !important;
    margin: auto;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue"),
    title="SafeSpace AI",
    css=custom_css
) as demo:

    gr.Markdown("# 🧠 SafeSpace – AI Mental Health Companion")

    gr.Markdown(f"""
<div style="background: #fee2e2; padding: 1rem; border-radius: 8px; border-left: 4px solid #dc2626;">
⚠️ <strong>CRISIS DISCLAIMER</strong><br>
This AI is a supportive tool, NOT a replacement for professional care.<br>
<strong>Emergency:</strong> 911 (US) | 988 (Suicide Prevention)<br>
<strong>Your Emergency Contact:</strong> {EMERGENCY_CONTACT}
</div>
""")

    # Chat interface
    chatbot = gr.Chatbot(
        value=show_welcome(),
        label="💬 Chat",
        height=500,
        avatar_images=("👤", "🤖")
    )

    audio_output = gr.Audio(label="🔊 Voice Response", autoplay=True)

    # Tabs
    with gr.Tabs():
        with gr.Tab("💬 Text"):
            msg = gr.Textbox(placeholder="What's on your mind?", lines=2, show_label=False)
            with gr.Row():
                submit = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear")

        with gr.Tab("🎤 Voice"):
            audio_input = gr.Audio(sources=["microphone"], type="filepath")
            voice_submit = gr.Button("🎤 Send Voice", variant="primary")

        with gr.Tab("📸 Emotion"):
            image_input = gr.Image(type="filepath")
            image_submit = gr.Button("📸 Analyze", variant="primary")

        with gr.Tab("🧍 Pose"):
            pose_input = gr.Image(type="filepath")
            with gr.Row():
                pose_submit = gr.Button("🧍 Analyze Pose")
                complete_submit = gr.Button("🎯 Complete Analysis")
            pose_output = gr.Image(label="Annotated")

        with gr.Tab("📊 Stats"):
            stats_display = gr.Markdown("Click Refresh")
            stats_button = gr.Button("🔄 Refresh")

        with gr.Tab("ℹ️ Help"):
            gr.Markdown("""
## 🆘 Crisis Resources
- 🚨 Emergency: 911
- ☎️ Suicide Prevention: 988
- 📱 Text: HOME to 741741

## 🎯 How to Use
- **Text Chat:** Type your thoughts
- **Voice:** Speak naturally
- **Emotion:** Upload selfie
- **Pose:** Upload full-body photo

All data saved to your Google Drive securely.
""")

    # Event handlers
    msg.submit(chat_with_ai, [msg, chatbot], [chatbot, msg, audio_output])
    submit.click(chat_with_ai, [msg, chatbot], [chatbot, msg, audio_output])
    voice_submit.click(process_audio_input, [audio_input, chatbot], [chatbot, msg, audio_output])
    image_submit.click(analyze_emotion_image, [image_input, chatbot], [chatbot, msg])
    pose_submit.click(analyze_pose_image, [pose_input, chatbot], [chatbot, msg, pose_output])
    complete_submit.click(analyze_complete_image, [pose_input, chatbot], [chatbot, msg, pose_output])
    stats_button.click(get_session_stats, outputs=[stats_display])
    clear_btn.click(clear_chat, outputs=[chatbot, audio_output])

# Launch
print("\n" + "="*70)
print("🎉 Launching SafeSpace (Direct Mode - No Backend)")
print("="*70)
print(f"✅ Storage: {BASE_DRIVE_PATH}")
print("="*70)

demo.launch(
    share=True,
    show_error=True,
    inbrowser=True
)

print("\n✅ SafeSpace is live!")

🚀 Starting Gradio interface (direct mode)...

🎉 Launching SafeSpace (Direct Mode - No Backend)
✅ Storage: /content/drive/MyDrive/SafeSpace_Data
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ac8f46d2132b6de5ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ SafeSpace is live!
